# Sentinel-2 monthly NDVI time-series (Earth Search)

Shows the **raster `aggregate=`** path: pull Sentinel-2 red + NIR over a small AOI for a couple of months, reduce per month into per-window COGs, then compute mean NDVI per month. The download runs live against Earth Search (anonymous).

In [ ]:
import tempfile

from earthlens.aggregate import AggregationConfig
from earthlens.core import EarthLens

out = tempfile.mkdtemp()
el = EarthLens(
    data_source="earth-search",
    start="2024-06-01",
    end="2024-07-31",
    dataset="sentinel-2-l2a",
    variables=["red", "nir"],
    aoi=[-3.72, 40.40, -3.67, 40.45],
    path=out,
    max_items=2,
)
monthly = el.download(
    aggregate=AggregationConfig(freq="1MS", op="mean", out_dir=out + "/monthly")
)
[p.name for p in monthly]

Each path is one COG per month, named `sentinel-2-l2a_mean_1MS_<YYYYMMDD>.tif`. Compute the mean NDVI per window:

In [ ]:
import numpy as np
from pyramids.dataset import Dataset

for cog in monthly:
    arr = Dataset.read_file(cog).read_array().astype(float)
    red, nir = arr[0], arr[1]
    ndvi = (nir - red) / (nir + red + 1e-9)
    print(cog.name, "mean NDVI:", round(float(np.nanmean(ndvi)), 3))